# Pandas vs Polars Benchmark

Based on: https://pipeline2insights.substack.com/p/pandas-vs-polars-benchmarking-dataframe

In [1]:
import os
import time
import psutil

import pandas as pd
import polars as pl
import numpy as np

In [2]:
def generate_benchmark_csv(
    output_path: str = "benchmark_data.csv",
    n_rows: int = 50_000_000,
    seed: int = 42,
    chunksize: int = 1_000_000
) -> str:
    """
    Genera un dataset simulado para benchmark entre Pandas y Polars y lo guarda en CSV.

    Parámetros:
    -----------
    output_path : str
        Ruta del archivo CSV de salida.
    n_rows : int
        Número total de filas a generar (por defecto 20 millones).
    seed : int
        Semilla aleatoria para reproducibilidad.
    chunksize : int
        Número de filas por bloque al escribir (para evitar alto consumo de memoria).

    Retorna:
    --------
    str
        Ruta completa del archivo CSV generado.
    """
    np.random.seed(seed)
    
    # Configurar rango de fechas
    start_date = np.datetime64("2023-01-01")
    end_date = np.datetime64("2024-01-01")
    total_days = (end_date - start_date).astype(int)

    # Eliminar archivo previo si existe
    if os.path.exists(output_path):
        os.remove(output_path)

    # Escritura por bloques (streaming para no saturar RAM)
    for start in range(0, n_rows, chunksize):
        end = min(start + chunksize, n_rows)
        size = end - start
        
        ids = np.arange(start + 1, end + 1)
        categories = np.random.choice(['A', 'B', 'C', 'D'], size=size)
        values = np.random.uniform(0, 100, size=size)
        timestamps = start_date + np.random.randint(0, total_days, size=size).astype('timedelta64[D]')
        
        df_chunk = pd.DataFrame({
            "id": ids,
            "category": categories,
            "value": values,
            "timestamp": timestamps
        })
        
        df_chunk.to_csv(output_path, mode='a', index=False, header=not os.path.exists(output_path))
        
        print(f"Chunk {start // chunksize + 1} ({size:,} filas) escrito...")

    print(f"\n✅ Archivo generado: {output_path}")
    return os.path.abspath(output_path)


In [3]:
generate_benchmark_csv(
    output_path = "benchmark_data.csv",
    n_rows = 25_000_000,
    seed = 42,
    chunksize = 1_000_000
)

Chunk 1 (1,000,000 filas) escrito...
Chunk 2 (1,000,000 filas) escrito...
Chunk 3 (1,000,000 filas) escrito...
Chunk 4 (1,000,000 filas) escrito...
Chunk 5 (1,000,000 filas) escrito...
Chunk 6 (1,000,000 filas) escrito...
Chunk 7 (1,000,000 filas) escrito...
Chunk 8 (1,000,000 filas) escrito...
Chunk 9 (1,000,000 filas) escrito...
Chunk 10 (1,000,000 filas) escrito...
Chunk 11 (1,000,000 filas) escrito...
Chunk 12 (1,000,000 filas) escrito...
Chunk 13 (1,000,000 filas) escrito...
Chunk 14 (1,000,000 filas) escrito...
Chunk 15 (1,000,000 filas) escrito...
Chunk 16 (1,000,000 filas) escrito...
Chunk 17 (1,000,000 filas) escrito...
Chunk 18 (1,000,000 filas) escrito...
Chunk 19 (1,000,000 filas) escrito...
Chunk 20 (1,000,000 filas) escrito...
Chunk 21 (1,000,000 filas) escrito...
Chunk 22 (1,000,000 filas) escrito...
Chunk 23 (1,000,000 filas) escrito...
Chunk 24 (1,000,000 filas) escrito...
Chunk 25 (1,000,000 filas) escrito...

✅ Archivo generado: benchmark_data.csv


'/Users/cesar/sandbox/ai_programming_foundations/notebooks/benchmark_data.csv'

In [4]:
csv_file="benchmark_data.csv"

In [5]:
def memory_usage() -> float:
    """
    Returns the current memory usage of the Python process in megabytes (MB).

    This function retrieves the Resident Set Size (RSS) — the portion of memory 
    occupied by a process that is held in RAM — for the current Python process 
    using the `psutil` library. It is useful for benchmarking and performance 
    monitoring tasks to evaluate how much memory a specific computation or 
    operation consumes.

    Returns
    -------
    float
        Memory usage of the current process in megabytes (MB).

    Notes
    -----
    - The function uses `psutil.Process(os.getpid())` to get the current process.
    - The returned value is calculated by dividing the RSS in bytes by (1024 * 1024)
      to convert it to megabytes.
    """
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

## 1. Loading DataFrame from CSV

In [6]:
# Pandas CSV Loading
print("Pandas CSV Loading")
start = time.time()
df_pandas = pd.read_csv(csv_file)
end = time.time()
print(f"Pandas: {end - start:.2f} sec, Memory: {memory_usage():.2f} MB")

Pandas CSV Loading
Pandas: 24.94 sec, Memory: 437.01 MB


In [7]:
# Polars CSV Loading
print("Polars CSV Loading")
start = time.time()
df_polars = pl.read_csv(csv_file)
end = time.time()
print(f"Polars: {end - start:.2f} sec, Memory: {memory_usage():.2f} MB")


Polars CSV Loading


Polars: 7.20 sec, Memory: 518.00 MB


## 2. Filtering Data

In [8]:
# Pandas Filtering Benchmark
num_runs = 10
pandas_times = []
print("Pandas Filtering Benchmark")
for _ in range(num_runs):
    start = time.time()
    df_pandas_filtered = df_pandas[df_pandas["value"] > 50]
    end = time.time()
    pandas_times.append(end - start)

pandas_avg_time = np.mean(pandas_times)
print(f"Pandas Avg Execution Time: {pandas_avg_time:.2f} sec over {num_runs} runs")


Pandas Filtering Benchmark
Pandas Avg Execution Time: 1.27 sec over 10 runs


In [9]:
# Polars Filtering Benchmark
num_runs = 10
polars_times = []
print("Polars Filtering Benchmark")
for _ in range(num_runs):
    start = time.time()
    df_polars_filtered = df_polars.filter(pl.col("value") > 50)
    end = time.time()
    polars_times.append(end - start)

polars_avg_time = np.mean(polars_times)
print(f"Polars Avg Execution Time: {polars_avg_time:.2f} sec over {num_runs} runs")


Polars Filtering Benchmark
Polars Avg Execution Time: 0.46 sec over 10 runs


## 3. Sorting

In [10]:
# Pandas Sorting Benchmark
pandas_times = []
print("Pandas Sorting Benchmark")
for _ in range(num_runs):
    start = time.time()
    df_pandas_sorted = df_pandas.sort_values("value", ascending=False)
    end = time.time()
    pandas_times.append(end - start)

pandas_avg_time = np.mean(pandas_times)
print(f"Pandas Avg Execution Time: {pandas_avg_time:.2f} sec over {num_runs} runs")


Pandas Sorting Benchmark
Pandas Avg Execution Time: 24.64 sec over 10 runs


In [11]:
# Polars Sorting Benchmark
polars_times = []
print("Polars Sorting Benchmark")
for _ in range(num_runs):
    start = time.time()
    df_polars_sorted = df_polars.sort("value", descending=True)
    end = time.time()
    polars_times.append(end - start)

polars_avg_time = np.mean(polars_times)
print(f"Polars Avg Execution Time: {polars_avg_time:.2f} sec over {num_runs} runs")


Polars Sorting Benchmark
Polars Avg Execution Time: 12.08 sec over 10 runs


## 4. Join

In [12]:
# Pandas Join Benchmark

# Create a smaller sample dataset for joining (1 million rows)
df_pandas_sample = df_pandas.sample(1_000_000, random_state=42)

pandas_times = []
print("Pandas Join Benchmark")
for _ in range(num_runs):
    start = time.time()
    df_pandas_joined = df_pandas.merge(df_pandas_sample, on="id", how="inner")
    end = time.time()
    pandas_times.append(end - start)

pandas_avg_time = np.mean(pandas_times)
print(f"Pandas Avg Execution Time: {pandas_avg_time:.2f} sec over {num_runs} runs")


Pandas Join Benchmark
Pandas Avg Execution Time: 10.17 sec over 10 runs


In [13]:
# Polars Join Benchmark

# Converting the exact dataset used for pandas sample into polars
df_polars_sample = pl.from_pandas(df_pandas_sample)

polars_times = []
print("Polars Join Benchmark")
for _ in range(num_runs):
    start = time.time()
    df_polars_joined = df_polars.join(df_polars_sample, on="id", how="inner")
    end = time.time()
    polars_times.append(end - start)

polars_avg_time = np.mean(polars_times)
print(f"Polars Avg Execution Time: {polars_avg_time:.2f} sec over {num_runs} runs")


Polars Join Benchmark
Polars Avg Execution Time: 1.18 sec over 10 runs
